# 特征缩放：7种缩放方法的对比与选择指南
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：01_数据预处理 | 源文件：特征缩放.py | 核心功能：StandardScaler、MinMaxScaler、RobustScaler 等 7 种缩放方法的完整演示

## 概述

想象一下：一个数据集里"年龄"的范围是 18-70，"收入"的范围是 20000-100000。如果直接喂给 KNN 或 SVM，模型会被"收入"这个大数值主导——不是因为收入更重要，仅仅是因为它的数字更大。

**特征缩放**（Feature Scaling）就是解决这个问题的。它把不同特征放到同一个"尺度"上，让模型不会因为数值范围的差异而产生偏见。

这个脚本用 Iris 数据集演示了 7 种主流缩放方法，每种方法的原理、适用场景和注意事项都有详细说明。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
# --- 继续 ---
    RobustScaler,
    MaxAbsScaler,
    Normalizer,
    PowerTransformer,
    QuantileTransformer,
# --- 继续 ---
)

## 数学原理

### 1. Z-score 标准化（StandardScaler）

**代码对应**：`StandardScaler().fit_transform(X)` 对每个特征计算：

$$x'_j = \frac{x_j - \mu_j}{\sigma_j}$$

其中 $\mu_j = \frac{1}{n}\sum_{i=1}^n x_{ij}$，$\sigma_j = \sqrt{\frac{1}{n}\sum_{i=1}^n (x_{ij} - \mu_j)^2}$。

**变换的数学性质**：

- 变换后 $\mathbb{E}[x'_j] = 0$，$\text{Var}(x'_j) = 1$
- 这是一个**仿射变换**（affine transformation），不改变数据的分布形状
- 如果原始数据服从正态分布 $\mathcal{N}(\mu, \sigma^2)$，变换后为标准正态 $\mathcal{N}(0, 1)$
- 各特征之间的相关系数不变：$\rho(x'_i, x'_j) = \rho(x_i, x_j)$

**为什么距离/梯度类模型需要标准化？** 以 KNN 为例，距离计算为：

$$d(\mathbf{x}, \mathbf{x'}) = \sqrt{\sum_{j=1}^{p} (x_j - x'_j)^2}$$

如果特征 $x_1 \in [0, 1000]$ 而 $x_2 \in [0, 1]$，则 $x_1$ 主导距离计算。标准化后各特征贡献相等。

### 2. Min-Max 归一化

**代码对应**：`MinMaxScaler(feature_range=(0, 1)).fit_transform(X)` 计算：

$$x'_{ij} = \frac{x_{ij} - \min_i(x_{ij})}{\max_i(x_{ij}) - \min_i(x_{ij})} \cdot (b - a) + a$$

其中 $[a, b]$ 为目标范围（默认 $[0, 1]$）。

**对异常值的敏感性分析**：设正常数据为 $\{1, 2, 3, 4, 5\}$，归一化后为 $\{0, 0.25, 0.5, 0.75, 1\}$。若加入异常值 100，数据变为 $\{1, 2, 3, 4, 5, 100\}$，归一化后正常值被压缩到 $[0, 0.04]$：

$$x'_{\text{normal}} = \frac{x - 1}{99} \in [0, 0.04]$$

这严重损失了正常数据的区分度。因此 MinMaxScaler 对异常值**极其敏感**。

### 3. 鲁棒缩放（RobustScaler）

**代码对应**：`RobustScaler().fit_transform(X)` 使用中位数和四分位距：

$$x'_{ij} = \frac{x_{ij} - \tilde{x}_j}{Q_3^{(j)} - Q_1^{(j)}}$$

**鲁棒性分析**：各种统计量的**崩溃点**（breakdown point）：

| 统计量 | 崩溃点 | 含义 |
|--------|--------|------|
| 均值 $\mu$ | 0% | 1 个极端值即可改变 |
| 中位数 $\tilde{x}$ | 50% | 可抵抗至半数数据被污染 |
| $Q_1, Q_3$ | 25% | 可抵抗 25% 数据被污染 |
| 标准差 $\sigma$ | 0% | 对异常值极敏感 |
| IQR | 25% | 较鲁棒 |

因此 RobustScaler 在含异常值的数据上比 StandardScaler 更可靠。

### 4. L2 归一化（Normalizer）

**代码对应**：`Normalizer(norm="l2").fit_transform(X)` 对**每个样本**做归一化：

$$\mathbf{x}'_i = \frac{\mathbf{x}_i}{\|\mathbf{x}_i\|_2} = \frac{\mathbf{x}_i}{\sqrt{\sum_{j=1}^p x_{ij}^2}}$$

注意与其他缩放器的关键区别：StandardScaler 和 MinMaxScaler 对**每个特征**（列）独立缩放，而 Normalizer 对**每个样本**（行）独立缩放。

变换后 $\|\mathbf{x}'_i\|_2 = 1$，所有样本落在单位超球面上。两个归一化样本的欧氏距离：

$$\|\mathbf{x}'_i - \mathbf{x}'_j\|_2^2 = 2(1 - \cos\theta_{ij})$$

其中 $\cos\theta_{ij} = \mathbf{x}'_i \cdot \mathbf{x}'_j$ 为余弦相似度。因此 L2 归一化后的欧氏距离**等价于余弦距离**，这在文本分类中非常有用。

### 5. Box-Cox 变换

**代码对应**：`PowerTransformer(method="box-cox").fit_transform(X)` 实现 Box-Cox 变换：

$$x' = \begin{cases} \dfrac{x^\lambda - 1}{\lambda} & \lambda \neq 0 \\[8pt] \ln(x) & \lambda = 0 \end{cases}$$

其中 $\lambda$ 通过**最大似然估计**选择，使变换后的数据最接近正态分布。

**MLE 求 $\lambda$ 的过程**：设变换后数据 $x' \sim \mathcal{N}(\mu, \sigma^2)$，对数似然函数为：

$$\ell(\lambda, \mu, \sigma^2) = -\frac{n}{2}\ln(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (x'_i - \mu)^2 + (\lambda - 1)\sum_{i=1}^n \ln x_i$$

最后一项是 Jacobian 修正项（从 $x$ 变换到 $x'$ 的体积变化）。对 $\lambda$ 进行网格搜索或数值优化求最大值。

**Box-Cox 的限制**：要求 $x > 0$。Yeo-Johnson 变换通过分段处理扩展到支持 $x \leq 0$：

$$x' = \begin{cases} \dfrac{(x+1)^\lambda - 1}{\lambda} & \lambda \neq 0,\; x \geq 0 \\[6pt] \ln(x+1) & \lambda = 0,\; x \geq 0 \\[6pt] -\dfrac{(-x+1)^{2-\lambda} - 1}{2-\lambda} & \lambda \neq 2,\; x < 0 \\[6pt] -\ln(-x+1) & \lambda = 2,\; x < 0 \end{cases}$$

### 6. 分位数变换（QuantileTransformer）

**代码对应**：`QuantileTransformer(output_distribution="normal").fit_transform(X)`。

**算法步骤**：

1. 对特征 $j$ 的原始值排序，计算每个值的经验分位数 $q_i = \hat{F}_j(x_{ij})$
2. 如果目标分布为正态，计算 $x'_{ij} = \Phi^{-1}(q_i)$，其中 $\Phi^{-1}$ 为标准正态的逆 CDF（PPF）

$$x'_{ij} = \Phi^{-1}\!\left(\hat{F}_j(x_{ij})\right)$$

该变换**单调递增**（保持排序），且将任意分布映射为精确的目标分布。代价是完全破坏了原始数据的线性关系和数值间距。

**近似最优性**：当样本量足够大时，经验 CDF $\hat{F}$ 收敛到真实 CDF $F$（Glivenko-Cantelli 定理），因此变换后的分布精确收敛到目标分布。

### 7. 缩放对正则化的影响

缩放直接影响正则化的效果。以 Lasso（L1 正则化线性回归）为例，目标函数为：

$$\min_{\mathbf{w}} \frac{1}{2n}\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha\|\mathbf{w}\|_1$$

如果特征 $x_1$ 的尺度是 $x_2$ 的 1000 倍，则 $w_1$ 的最优值约为 $w_2$ 的 $1/1000$。L1 惩罚 $\alpha|w_1| + \alpha|w_2|$ 会不公平地惩罚 $w_2$。

标准化后各特征尺度相同，正则化对所有特征**一视同仁**。这就是为什么使用正则化的线性模型（Ridge、Lasso、ElasticNet）**必须**先做特征缩放。

**数学推导**：设特征 $j$ 缩放因子为 $s_j$，即 $x'_{ij} = x_{ij}/s_j$。则缩放后的系数 $w'_j = s_j w_j$。L1 惩罚变为：

$$\alpha \sum_j |w'_j| = \alpha \sum_j s_j |w_j|$$

若所有 $s_j$ 相等（标准化后），惩罚对所有 $w_j$ 公平。

## 代码结构

| 缩放方法 | 核心公式 | 适用场景 |
|----------|---------|----------|
| StandardScaler | (x - μ) / σ | 默认首选 |
| MinMaxScaler | (x - min) / (max - min) | 需要固定范围 |
| RobustScaler | (x - median) / IQR | 有异常值 |
| MaxAbsScaler | x / |max| | 稀疏数据 |
| Normalizer | x / ||x|| | 文本分类 |
| PowerTransformer | Box-Cox / Yeo-Johnson | 偏态分布 |
| QuantileTransformer | 分位数映射 | 极端偏态/异常值 |

### 1. 加载示例数据

运行 1. 加载示例数据 的代码，观察算法在该环节的行为。

In [ ]:
iris = load_iris()
X = iris.data
feature_names = iris.feature_names
print("=== 原始数据统计 ===")
print(f"特征: {feature_names}")
# --- 输出结果 ---
print(f"各特征范围:")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{X[:, i].min():.1f}, {X[:, i].max():.1f}], "
          f"均值={X[:, i].mean():.2f}, 标准差={X[:, i].std():.2f}")

### 2. Z-score 标准化（StandardScaler）

运行 2. Z-score 标准化（StandardScaler） 的代码，观察算法在该环节的行为。

In [ ]:
# (x - μ) / σ，使每个特征均值为 0，标准差为 1
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)
print(f"\n=== StandardScaler (Z-score 标准化) ===")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{X_std[:, i].min():.2f}, {X_std[:, i].max():.2f}], "
# --- 继续 ---
          f"均值={X_std[:, i].mean():.4f}, 标准差={X_std[:, i].std():.4f}")

### 3. Min-Max 归一化（MinMaxScaler）

运行 3. Min-Max 归一化（MinMaxScaler） 的代码，观察算法在该环节的行为。

In [ ]:
# (x - min) / (max - min)，缩放到 [0, 1]
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(X)
print(f"\n=== MinMaxScaler (归一化到 [0,1]) ===")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{X_mm[:, i].min():.2f}, {X_mm[:, i].max():.2f}]")

# 也可指定范围，如 [-1, 1]
scaler_mm2 = MinMaxScaler(feature_range=(-1, 1))
X_mm2 = scaler_mm2.fit_transform(X)
print(f"\n=== MinMaxScaler (归一化到 [-1,1]) ===")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{X_mm2[:, i].min():.2f}, {X_mm2[:, i].max():.2f}]")

### 4. 鲁棒缩放（RobustScaler）

运行 4. 鲁棒缩放（RobustScaler） 的代码，观察算法在该环节的行为。

In [ ]:
# 使用中位数和 IQR 缩放，对异常值不敏感
scaler_robust = RobustScaler()
X_robust = scaler_robust.fit_transform(X)
print(f"\n=== RobustScaler (基于中位数和 IQR) ===")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{X_robust[:, i].min():.2f}, {X_robust[:, i].max():.2f}], "
# --- 数值计算 ---
          f"中位数={np.median(X_robust[:, i]):.4f}")

### 5. 最大绝对值缩放（MaxAbsScaler）

运行 5. 最大绝对值缩放（MaxAbsScaler） 的代码，观察算法在该环节的行为。

In [ ]:
# x / |max|，缩放到 [-1, 1]，适合稀疏数据
scaler_abs = MaxAbsScaler()
X_abs = scaler_abs.fit_transform(X)
print(f"\n=== MaxAbsScaler (最大绝对值缩放) ===")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{X_abs[:, i].min():.2f}, {X_abs[:, i].max():.2f}]")

### 6. L2 归一化（Normalizer）

运行 6. L2 归一化（Normalizer） 的代码，观察算法在该环节的行为。

In [ ]:
# 将每个样本缩放为单位范数（L2 范数为 1），适合文本分类等
normalizer = Normalizer(norm="l2")
X_norm = normalizer.fit_transform(X)
print(f"\n=== Normalizer (L2 归一化) ===")
print(f"每个样本的 L2 范数（前 5 个）: {np.linalg.norm(X_norm[:5], axis=1).round(4)}")

### 7. 幂变换（PowerTransformer）

运行 7. 幂变换（PowerTransformer） 的代码，观察算法在该环节的行为。

In [ ]:
# 使数据更接近正态分布（Yeo-Johnson 或 Box-Cox）
pt_yj = PowerTransformer(method="yeo-johnson")
X_yj = pt_yj.fit_transform(X)
print(f"\n=== PowerTransformer (Yeo-Johnson) ===")
for i, name in enumerate(feature_names):
    print(f"  {name}: 均值={X_yj[:, i].mean():.4f}, 标准差={X_yj[:, i].std():.4f}")

# Box-Cox 要求数据为正数
pt_bc = PowerTransformer(method="box-cox")
X_bc = pt_bc.fit_transform(X)  # Iris 数据均为正数
print(f"\n=== PowerTransformer (Box-Cox) ===")
for i, name in enumerate(feature_names):
    print(f"  {name}: 均值={X_bc[:, i].mean():.4f}, 标准差={X_bc[:, i].std():.4f}")

### 8. 分位数变换（QuantileTransformer）

运行 8. 分位数变换（QuantileTransformer） 的代码，观察算法在该环节的行为。

In [ ]:
# 映射到均匀分布或正态分布，对异常值最鲁棒
qt_norm = QuantileTransformer(output_distribution="normal", random_state=42)
X_qt = qt_norm.fit_transform(X)
print(f"\n=== QuantileTransformer (映射到正态分布) ===")
for i, name in enumerate(feature_names):
    print(f"  {name}: 均值={X_qt[:, i].mean():.4f}, 标准差={X_qt[:, i].std():.4f}")

### 9. 缩放方法选择指南

运行 9. 缩放方法选择指南 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 缩放方法选择指南 ===")
print("1. StandardScaler: 默认首选，适合大多数算法（SVM/KNN/PCA/逻辑回归）")
print("2. MinMaxScaler: 需要固定范围时使用（神经网络、图像像素）")
print("3. RobustScaler: 数据含较多异常值时使用")
print("4. MaxAbsScaler: 稀疏数据保持稀疏性")
# --- 输出结果 ---
print("5. Normalizer: 文本分类、余弦相似度计算")
print("6. PowerTransformer: 特征严重偏态时，先正态化再缩放")
print("7. QuantileTransformer: 异常值极多或分布极偏时使用")

# 注意事项
print("\n=== 注意事项 ===")
print("- 缩放器应在训练集上 fit，然后分别 transform 训练集和测试集")
print("- 避免数据泄露：绝不能用测试集信息来 fit 缩放器")
print("- 树模型（决策树/随机森林/XGBoost）通常不需要特征缩放")
print("- 距离-based 模型（KNN/SVM/线性回归/逻辑回归）强烈建议缩放")

## 关键代码解释

### StandardScaler —— 最常用的默认选择

```python
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)
```

将每个特征变换为均值 0、标准差 1 的分布。**数学本质**：Z-score 标准化。适合大多数基于距离或梯度的算法：SVM、KNN、逻辑回归、PCA、神经网络。

**注意**：StandardScaler 不改变分布的形状，只是平移和缩放。如果原始分布是偏态的，标准化后依然是偏态的。

### MinMaxScaler —— 归一化到固定范围

```python
scaler_mm = MinMaxScaler(feature_range=(-1, 1))
```

将数据线性映射到指定范围（默认 [0, 1]）。**最适合**：图像像素（0-255 → 0-1）、神经网络输入（需要有界激活函数如 sigmoid）。

**致命弱点**：对异常值极其敏感。一个极端的异常值会压缩正常数据的范围。比如数据是 [1, 2, 3, 100]，归一化后正常值全挤在 [0, 0.02] 区间。

### RobustScaler —— 异常值的克星

```python
scaler_robust = RobustScaler()
X_robust = scaler_robust.fit_transform(X)
```

用**中位数**和**四分位距（IQR）**代替均值和标准差。中位数和 IQR 对异常值天然鲁棒——即使有几个极端值，中位数几乎不变。

**适用场景**：数据中包含较多异常值，或者特征分布严重偏态。

### PowerTransformer —— 让偏态变正态

```python
pt_yj = PowerTransformer(method="yeo-johnson")
pt_bc = PowerTransformer(method="box-cox")
```

Box-Cox 变换：x' = (x^λ - 1) / λ（λ 由数据自动优化），要求数据为正数。Yeo-Johnson 是 Box-Cox 的扩展版，支持负数和零。

**核心价值**：很多统计方法和机器学习算法假设数据近似正态分布。PowerTransformer 能让严重偏态的特征（如收入、房价）更接近正态。

### QuantileTransformer —— 最激进的变换

```python
qt_norm = QuantileTransformer(output_distribution="normal", random_state=42)
```

将数据映射到均匀分布或正态分布。**原理**：先计算每个值的分位数，再映射到目标分布。对异常值**最鲁棒**，但会严重改变数据的原始结构。

**使用场景**：异常值极多、分布极偏，其他方法都搞不定的时候。

## 使用示例

```python
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# 在 Pipeline 中安全使用缩放
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])
pipe.fit(X_train, y_train)        # 缩放器只在训练集上 fit
score = pipe.score(X_test, y_test) # 测试集自动用训练集的参数 transform
```

## 注意事项

1. **先 fit 后 transform**：
it 只能在训练集上调用，	ransform 分别应用到训练集和测试集。用 
it_transform 一步到位可以，但只限于训练集
2. **Pipeline 是最佳实践**：把缩放器放进 Pipeline，可以避免忘记 transform 测试集的低级错误
3. **树模型不需要缩放**：决策树、随机森林、XGBoost 等基于分裂点的模型不受特征尺度影响
4. **距离/梯度模型必须缩放**：KNN、SVM、线性回归、逻辑回归、PCA、神经网络都强烈建议缩放
5. **不要在交叉验证外 fit**：如果做交叉验证，缩放器应该在每一折的训练集上重新 fit

## 总结与延伸

以上代码展示了 **特征缩放** 的完整流程。

**解读要点**：
- 关注 **特征重要性** 排名和分布
- 对比特征选择前后的模型性能
- 观察特征交叉是否带来性能提升

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **为什么 Pipeline 这么重要？** 因为它把预处理和模型封装在一起，确保训练和推理执行完全相同的操作，杜绝数据泄露
- **特征缩放 vs 特征选择**：缩放不会改变特征的重要性排序（对线性模型除外），但会影响基于正则化的特征选择（如 Lasso）
- **自适应缩放**：某些在线学习场景需要动态更新缩放参数，sklearn 的 PartialFit 支持增量更新
- **sklearn 的 ColumnTransformer**：可以对不同列应用不同的缩放策略——数值列用 StandardScaler，分类列用 OneHotEncoder
